# Multi-Quantile Prediction Intervals
# 多分位数预测区间

PipelineTS supports outputting prediction intervals at **multiple coverage levels** simultaneously.
PipelineTS 支持同时输出**多个覆盖水平**的预测区间。

This tutorial covers:
本教程涵盖：

1. **Standard single-quantile intervals / 标准单分位数区间**
2. **Multi-quantile output with `predict_quantiles()` / 多分位数输出**
3. **Conformal Prediction and CQR / 保形预测与 CQR**
4. **Monotonicity guarantees / 单调性保证**
5. **Visualization of multi-quantile bands / 多分位数带可视化**
6. **SmartRouter multi-quantile / SmartRouter 多分位数**

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Prepare example data / 准备示例数据
np.random.seed(42)
n = 200
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
values = 50 + 10 * np.sin(np.linspace(0, 6 * np.pi, n)) + np.random.randn(n) * 3
data = pd.DataFrame({'date': dates, 'value': values})

LAGS = 12
PREDICT_N = 15

print(f"Data shape / 数据形状: {data.shape}")
data.head()

## 1. Standard Single-Quantile Interval
## 1. 标准单分位数区间

Set `quantile=0.9` to get a 90% prediction interval. The output contains `value_lower` and `value_upper` columns.

设置 `quantile=0.9` 获取 90% 预测区间。输出包含 `value_lower` 和 `value_upper` 列。

In [ ]:
from PipelineTS.ml_model import TorchBoostingForestModel

model = TorchBoostingForestModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9,  # 90% coverage / 90% 覆盖率
)
model.fit(data)
result = model.predict(PREDICT_N)

print("Columns / 列名:", result.columns.tolist())
result

In [ ]:
from PipelineTS.plot import plot_forecast

plot_forecast(
    data, result,
    time_col='date', target_col='value',
    history_tail=50,
    title='90% 预测区间 (单分位数)',
    lang='zh',
)

## 2. Multi-Quantile Output
## 2. 多分位数输出

Use `predict_quantiles()` on `ModelPipeline` to output intervals at multiple coverage levels simultaneously.

在 `ModelPipeline` 上使用 `predict_quantiles()` 同时输出多个覆盖水平的区间。

In [ ]:
from PipelineTS.pipeline import ModelPipeline

pipeline = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9,
    include_models=['torch_boosting_forest', 'torch_bagging_forest'],
    cv=2,
)
pipeline.fit(data)
print("Best model / 最佳模型:", pipeline.leader_board_.iloc[0]['model'])

In [ ]:
# Multi-quantile prediction at 50%, 80%, 95% coverage
# 多分位数预测：50%、80%、95% 覆盖率
result_mq = pipeline.predict_quantiles(n=PREDICT_N, levels=[0.5, 0.8, 0.95])

print("Columns / 列名:")
for col in result_mq.columns:
    print(f"  {col}")
result_mq

In [ ]:
# Visualize multi-quantile bands (auto-detected by plot_forecast)
# 可视化多分位数带（plot_forecast 自动检测）
plot_forecast(
    data, result_mq,
    time_col='date', target_col='value',
    history_tail=50,
    title='多分位数预测区间 (50%, 80%, 95%)',
    lang='zh',
)

## 3. How It Works: Conformal Prediction & CQR
## 3. 工作原理：保形预测与 CQR

PipelineTS uses two interval estimation methods:
PipelineTS 使用两种区间估计方法：

### Conformal Prediction (for ML/Statistical models)
### 保形预测（用于 ML/统计模型）

Based on signed residuals from cross-validation. Given coverage level α:
基于交叉验证的带符号残差。给定覆盖水平 α：

- Lower bound = prediction - quantile(residuals, α/2)
- Upper bound = prediction + quantile(residuals, 1 - α/2)

### Conformalized Quantile Regression (CQR, for NN models)
### 保形分位数回归（CQR，用于神经网络模型）

Combines quantile regression with conformal calibration:
结合分位数回归和保形校准：

- Model learns lower/median/upper quantile heads simultaneously
- 模型同时学习下/中/上分位数头
- Conformal correction Q_hat provides coverage guarantee
- 保形校正 Q_hat 提供覆盖率保证
- Intervals are **adaptive**: wider where model is uncertain
- 区间是**自适应的**：模型不确定的地方更宽

## 4. Monotonicity Guarantee
## 4. 单调性保证

Wider coverage levels **always** produce wider intervals. Let's verify.

更宽的覆盖水平**始终**产生更宽的区间。让我们验证。

In [ ]:
# Verify monotonicity: 95% interval >= 80% interval >= 50% interval
# 验证单调性：95% 区间 >= 80% 区间 >= 50% 区间

levels = [0.5, 0.8, 0.95]
for i in range(1, len(levels)):
    narrow = levels[i-1]
    wide = levels[i]

    narrow_width = (
        result_mq[f'value_q{narrow}_upper'] - result_mq[f'value_q{narrow}_lower']
    ).values
    wide_width = (
        result_mq[f'value_q{wide}_upper'] - result_mq[f'value_q{wide}_lower']
    ).values

    all_monotonic = np.all(wide_width >= narrow_width - 1e-6)
    print(f"q{wide} >= q{narrow}: {all_monotonic}")
    print(f"  {narrow} width: mean={narrow_width.mean():.2f}")
    print(f"  {wide} width:   mean={wide_width.mean():.2f}")

## 5. CQR with Neural Networks
## 5. 神经网络的 CQR 区间

Neural network models use CQR for adaptive intervals.

神经网络模型使用 CQR 获得自适应区间。

In [ ]:
from PipelineTS.nn_model import NLinearModel

nn_model = NLinearModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9,
    epochs=50, patience=10, verbose=False
)
nn_model.fit(data)
nn_result = nn_model.predict(PREDICT_N)

print("NN prediction with CQR intervals / 神经网络 CQR 区间预测:")
print(f"Columns: {nn_result.columns.tolist()}")
nn_result

In [ ]:
plot_forecast(
    data, nn_result,
    time_col='date', target_col='value',
    history_tail=50,
    title='NLinear CQR 预测区间',
    lang='zh',
)

## 6. SmartRouter Multi-Quantile
## 6. SmartRouter 多分位数

`SmartRouter` also supports `predict_quantiles()`.

`SmartRouter` 也支持 `predict_quantiles()`。

In [ ]:
from PipelineTS.pipeline import SmartRouter

router = SmartRouter(
    time_col='date', target_col='value',
    quantile=0.9,
    max_models=3,
)
router.fit(data)

# Multi-quantile via SmartRouter
# 通过 SmartRouter 进行多分位数预测
router_mq = router.predict_quantiles(n=PREDICT_N, levels=[0.5, 0.9])

print("SmartRouter multi-quantile columns / SmartRouter 多分位数列:")
for col in router_mq.columns:
    print(f"  {col}")
router_mq.head()

## Summary / 总结

| Feature / 功能 | API | Description / 描述 |
|---|---|---|
| Single quantile / 单分位数 | `quantile=0.9` | 90% prediction interval / 90% 预测区间 |
| Multi-quantile / 多分位数 | `predict_quantiles(n, levels)` | Multiple intervals simultaneously / 同时输出多个区间 |
| Conformal / 保形预测 | ML & Statistical models | Based on signed residuals / 基于带符号残差 |
| CQR | NN models / 神经网络模型 | Adaptive intervals with coverage guarantee / 自适应区间 + 覆盖率保证 |
| Monotonicity / 单调性 | All models / 所有模型 | Wider coverage → wider intervals / 更宽覆盖率 → 更宽区间 |